In [1]:
#imports

#from tensorzinb.tensorzinb import TensorZINB
import scMPRAforge as scm

2026-01-07 09:56:42.883640: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-07 09:56:42.886978: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
import pandas as pd
import numpy as np
import time
import pickle
from formulaic import Formula
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#create dask cluster

from dask_jobqueue import SLURMCluster
from dask.distributed import Client

cluster=SLURMCluster(
    cores=4,#cores per slurm job
    memory="512G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p day", 
        f"--job-name=simclust_worker",
        f"--time=6:00:00",
        f"--output=worker_%j.out"]
)

cluster.scale(jobs=1)

client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="20s"  # Worker heartbeat interval
    )

#from dask.distributed import Client, LocalCluster
#cluster=LocalCluster(memory_limit='8GB')
#client = Client(cluster)

2026-01-07 14:23:00,520 - distributed.scheduler - ERROR - Task _smart_matrix-e860d35924fef76bfa39f9c58d1bf099 marked as failed because 4 workers died while trying to run it
2026-01-07 14:23:00,528 - distributed.scheduler - ERROR - Task _smart_matrix-3b242be966e81709dea47ff07341574b marked as failed because 4 workers died while trying to run it


In [5]:
client.dashboard_link

'http://10.18.22.66:39315/status'

# Describe with Ortho


In [6]:
data_root="/home/sxl6/project_pi_skr2/sxl6/tabula_data/seelig"
path= data_root
name="ortho_test_seelig"

seelig=scm.scMPRA_data.from_tsv(f"{data_root}/susanna_seelig_counts_grouped.txt")

In [7]:
seelig.set_negative_controls(["AACGCCCTCCACGGATGGGCCGGCCAATAAGAAGCGTTAGCGGACTCATGCGTTACGCGCCTCCGAGTTATGGGGGGGGAGGCGCGTATCTCGTGGAGAAGAAGCGATGTAACGCTTGGGCGATAAGCTTATAAGGAAGATATTT",
    "CCCTCGGAGTTAATAAGATACGCGGATCGATATCGGCTTGAAGAAGCGTATCTTATCTTCAGATGGGGATGTCGCGCATCCACCCAGTGGGCACCGCCGCTATAGAAGGGTGATAACGCTTCTCAGCCTTCAGGCTCTGGGTCTT"])
seelig.set_reference_cell("HEPG2")
seelig.ortho_filter()

scMPRAforge: INFO: Dropped 62 of 2688 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [8]:
primordial=scm.ortho()
primordial.criss_cross(client=client,
                       dat=seelig)
primordial.extract_params(client)
primordial.save(path,name)

KilledWorker: Attempted to run task '_smart_matrix-e860d35924fef76bfa39f9c58d1bf099' on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://10.18.22.80:44571. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [ ]:
print("finished!")

In [ ]:
client.close()
cluster.close()